# TRIBE v2 — Pipeline Benchmark

This notebook measures the wall-clock time of every major phase in the TRIBE v2 pipeline to establish baselines for optimization.

**Requirements:**
- GPU runtime (Menu → Runtime → Change runtime type → T4 GPU)
- HuggingFace account with access to [LLaMA 3.2-3B](https://huggingface.co/meta-llama/Llama-3.2-3B)

**Output:** A `benchmark_results.json` file with per-phase timings, GPU info, and memory snapshots.

## 0. Benchmark utilities

In [ ]:
import time
import json
import subprocess
from pathlib import Path
from dataclasses import dataclass, field, asdict
from contextlib import contextmanager


@dataclass
class PhaseResult:
    """Timing + memory snapshot for a single pipeline phase."""
    name: str
    elapsed_s: float = 0.0
    gpu_mem_allocated_mb: float = 0.0
    gpu_mem_reserved_mb: float = 0.0
    notes: str = ""


@dataclass
class BenchmarkResults:
    """Collects all phase results + system info."""
    gpu_name: str = ""
    gpu_vram_mb: float = 0.0
    cuda_version: str = ""
    torch_version: str = ""
    video_duration_s: float = 0.0
    video_file: str = ""
    phases: list = field(default_factory=list)

    def add(self, result: PhaseResult):
        self.phases.append(asdict(result))
        print(f"  ✓ {result.name}: {result.elapsed_s:.2f}s"
              f"  (GPU: {result.gpu_mem_allocated_mb:.0f} MB alloc, "
              f"{result.gpu_mem_reserved_mb:.0f} MB reserved)")

    def summary(self):
        total = sum(p["elapsed_s"] for p in self.phases)
        print("\n" + "="*60)
        print(f"BENCHMARK SUMMARY  ({self.gpu_name}, {self.gpu_vram_mb:.0f} MB VRAM)")
        print("="*60)
        for p in self.phases:
            pct = (p["elapsed_s"] / total * 100) if total > 0 else 0
            bar = "█" * int(pct / 2)
            print(f"  {p['name']:<35s} {p['elapsed_s']:>8.2f}s  ({pct:5.1f}%)  {bar}")
        print(f"  {'TOTAL':<35s} {total:>8.2f}s")
        print("="*60)

    def to_dict(self):
        return asdict(self)


def gpu_mem_snapshot():
    """Return (allocated_mb, reserved_mb) on the current CUDA device."""
    try:
        import torch
        if torch.cuda.is_available():
            return (
                torch.cuda.memory_allocated() / 1e6,
                torch.cuda.memory_reserved() / 1e6,
            )
    except Exception:
        pass
    return (0.0, 0.0)


@contextmanager
def timed_phase(bench: BenchmarkResults, name: str, notes: str = ""):
    """Context manager that times a phase and records GPU memory."""
    print(f"\n▶ {name} ...")
    start = time.perf_counter()
    yield
    elapsed = time.perf_counter() - start
    alloc, reserved = gpu_mem_snapshot()
    bench.add(PhaseResult(
        name=name,
        elapsed_s=round(elapsed, 3),
        gpu_mem_allocated_mb=round(alloc, 1),
        gpu_mem_reserved_mb=round(reserved, 1),
        notes=notes,
    ))


bench = BenchmarkResults()
print("Benchmark utilities loaded.")

## 1. Install dependencies

In [ ]:
with timed_phase(bench, "1. Install dependencies"):
    !pip install "numpy<2.1" -q
    !pip install "tribev2[plotting] @ git+https://github.com/kingjulio8238/tribev2.git" -q

## 2. Collect system info

In [ ]:
import torch

bench.torch_version = torch.__version__
bench.cuda_version = torch.version.cuda or "N/A"

if torch.cuda.is_available():
    bench.gpu_name = torch.cuda.get_device_name(0)
    bench.gpu_vram_mb = torch.cuda.get_device_properties(0).total_memory / 1e6
    print(f"GPU: {bench.gpu_name}")
    print(f"VRAM: {bench.gpu_vram_mb:.0f} MB")
    print(f"CUDA: {bench.cuda_version}")
else:
    print("WARNING: No GPU detected! Benchmarks will be very slow.")

print(f"PyTorch: {bench.torch_version}")

# Also capture nvidia-smi for the record
!nvidia-smi

## 3. Authenticate with HuggingFace

In [ ]:
from huggingface_hub import login
login()

## 4. Download sample video

In [ ]:
from tribev2.demo_utils import download_file

CACHE_FOLDER = Path("./cache")
CACHE_FOLDER.mkdir(exist_ok=True)
video_path = CACHE_FOLDER / "sintel_trailer.mp4"

with timed_phase(bench, "2. Download video"):
    download_file(
        "https://download.blender.org/durian/trailer/sintel_trailer-480p.mp4",
        video_path,
    )

# Record video duration
from moviepy import VideoFileClip
clip = VideoFileClip(str(video_path))
bench.video_duration_s = clip.duration
bench.video_file = video_path.name
print(f"Video: {video_path.name}, {clip.duration:.1f}s, {clip.size}")
clip.close()

## 5. Load model (checkpoint download + init)

In [ ]:
from tribev2.demo_utils import TribeModel

with timed_phase(bench, "3. Model load (checkpoint + init)",
                 notes="Downloads ~709MB checkpoint on first run, inits config"):
    model = TribeModel.from_pretrained(
        "facebook/tribev2",
        cache_folder=CACHE_FOLDER,
        device="auto",
    )
print("Model loaded successfully")

## 6. Build events (audio extraction + WhisperX transcription)

In [ ]:
with timed_phase(bench, "4. Build events (audio + WhisperX)",
                 notes="ExtractAudioFromVideo, ChunkEvents, WhisperX, AddText/Context"):
    df = model.get_events_dataframe(video_path=video_path)

print(f"Events: {len(df)} rows")
print(f"Types: {df['type'].value_counts().to_dict()}")
display(df.head(8)[["type", "start", "duration", "text", "context"]])

## 7. Feature extraction (per-modality, instrumented)

This is where the bulk of compute happens. We call `get_loaders` which internally runs `extractor.prepare(events)` for each modality. We monkey-patch to time each extractor individually.

In [ ]:
import gc
from neuralset.events.utils import standardize_events
from neuralset.events.etypes import EventTypesHelper
import neuralset as ns
import numpy as np

# We replicate the get_loaders logic but instrument each extractor.prepare() call
events = standardize_events(df)

# Build extractors dict (same logic as Data.get_loaders)
extractors = {}
for modality in model.data.features_to_use:
    extractors[modality] = getattr(model.data, f"{modality}_feature")
extractors["subject_id"] = model.data.subject_id

# Add dummy trigger events (same as get_loaders)
dummy_events = []
for timeline_name, timeline in events.groupby("timeline"):
    split = "all"
    dummy_event = {
        "type": "CategoricalEvent",
        "timeline": timeline_name,
        "start": timeline.start.min(),
        "duration": timeline.stop.max() - timeline.start.min(),
        "split": split,
        "subject": timeline.subject.unique()[0],
    }
    dummy_events.append(dummy_event)
import pandas as pd
events_with_triggers = pd.concat([events, pd.DataFrame(dummy_events)])
events_with_triggers = standardize_events(events_with_triggers)

# Remove extractors with no matching events
to_remove = set()
for name, ext in extractors.items():
    event_types = EventTypesHelper(ext.event_types).names
    if not any(et in events_with_triggers.type.unique() for et in event_types):
        to_remove.add(name)
for name in to_remove:
    del extractors[name]
    print(f"  Skipping extractor '{name}' (no matching events)")

print(f"\nExtractors to benchmark: {list(extractors.keys())}")

In [ ]:
from tribev2.main import _free_extractor_model

# Time each extractor.prepare() individually
for name, extractor in extractors.items():
    phase_name = f"5. Feature extraction: {name}"
    notes = f"extractor type: {type(extractor).__name__}"
    with timed_phase(bench, phase_name, notes=notes):
        extractor.prepare(events_with_triggers)
    # Free GPU memory after each extractor (like the real pipeline does)
    if name != "subject_id":
        _free_extractor_model(extractor)

print("\nAll feature extraction complete.")

## 8. Build dataloader

In [ ]:
with timed_phase(bench, "6. Build dataloader",
                 notes="Segment creation + SegmentDataset init"):
    TR = model.data.TR
    segments = ns.segments.list_segments(
        events_with_triggers,
        triggers=events_with_triggers.type == "CategoricalEvent",
        stride=(model.data.duration_trs - model.data.overlap_trs_train) * TR,
        duration=model.data.duration_trs * TR,
        stride_drop_incomplete=model.data.stride_drop_incomplete,
    )
    dataset = ns.dataloader.SegmentDataset(
        extractors=extractors,
        segments=segments,
        remove_incomplete_segments=False,
    )
    loader = dataset.build_dataloader(
        shuffle=False,
        num_workers=model.data.num_workers,
        batch_size=model.data.batch_size,
    )

print(f"Dataloader: {len(loader)} batches, batch_size={model.data.batch_size}")

## 9. Model inference (forward pass)

In [ ]:
from einops import rearrange
from tqdm import tqdm

brain_model = model._model

with timed_phase(bench, "7. Model inference (forward pass)",
                 notes="Transformer forward pass over all batches"):
    preds_list, all_segments = [], []
    n_samples, n_kept = 0, 0
    with torch.inference_mode():
        for batch in tqdm(loader):
            batch = batch.to(brain_model.device)
            batch_segments = []
            for segment in batch.segments:
                for t in np.arange(0, segment.duration - 1e-2, TR):
                    batch_segments.append(
                        segment.copy(offset=t, duration=TR)
                    )
            keep = np.array([len(s.ns_events) > 0 for s in batch_segments])
            n_kept += keep.sum()
            n_samples += len(batch_segments)
            batch_segments = [s for i, s in enumerate(batch_segments) if keep[i]]
            y_pred = brain_model(batch).detach().cpu().numpy()
            y_pred = rearrange(y_pred, "b d t -> (b t) d")[keep]
            preds_list.append(y_pred)
            all_segments.extend(batch_segments)

    preds = np.concatenate(preds_list)

print(f"Predictions: {preds.shape} ({n_kept}/{n_samples} segments kept)")

## 10. Normalization

In [ ]:
from tribev2.plotting.utils import robust_normalize

with timed_phase(bench, "8. Normalization (robust_normalize)",
                 notes="Per-vertex 99th percentile normalization"):
    preds_norm = robust_normalize(preds, percentile=99).astype(np.float32)

print(f"Normalized predictions: {preds_norm.shape}, "
      f"range [{preds_norm.min():.3f}, {preds_norm.max():.3f}]")

## 11. Export (mesh + predictions + stimulus metadata)

In [ ]:
import nibabel as nib
from nilearn.datasets import fetch_surf_fsaverage
import shutil

OUTPUT_DIR = Path("./viewer_data")

with timed_phase(bench, "9a. Export mesh",
                 notes="fsaverage5 vertices, faces, sulcal depth, HCP parcellation"):
    mesh_dir = OUTPUT_DIR / "mesh"
    mesh_dir.mkdir(parents=True, exist_ok=True)

    fsaverage = fetch_surf_fsaverage("fsaverage5")
    all_coords, all_faces, all_sulc = [], [], []
    vertex_offset = 0
    GAP = 3.0
    for hemi in ("left", "right"):
        pial = nib.load(getattr(fsaverage, f"pial_{hemi}")).darrays
        pial_coords, faces = pial[0].data, pial[1].data
        infl_coords = nib.load(getattr(fsaverage, f"infl_{hemi}")).darrays[0].data
        coords = 0.5 * pial_coords + 0.5 * infl_coords
        if hemi == "left":
            coords[:, 0] -= coords[:, 0].max() + GAP
        else:
            coords[:, 0] -= coords[:, 0].min() - GAP
        sulc = nib.load(getattr(fsaverage, f"sulc_{hemi}")).darrays[0].data
        all_coords.append(coords)
        all_faces.append(faces + vertex_offset)
        all_sulc.append(sulc)
        vertex_offset += coords.shape[0]

    vertices = np.concatenate(all_coords).astype(np.float32)
    faces_arr = np.concatenate(all_faces).astype(np.uint32)
    sulcal = np.concatenate(all_sulc).astype(np.float32)
    vertices.tofile(str(mesh_dir / "vertices.bin"))
    faces_arr.tofile(str(mesh_dir / "faces.bin"))
    sulcal.tofile(str(mesh_dir / "sulcal_depth.bin"))

    # HCP parcellation
    try:
        from tribev2.utils import get_hcp_labels
        labels_dict = get_hcp_labels(mesh="fsaverage5", combine=False, hemi="both")
        roi_names = ["unknown"] + sorted(labels_dict.keys())
        roi_map = {n: i for i, n in enumerate(roi_names)}
        parc = np.zeros(vertices.shape[0], dtype=np.uint16)
        for name, verts in labels_dict.items():
            for v in verts:
                if v < parc.shape[0]:
                    parc[int(v)] = roi_map[name]
        parc.tofile(str(mesh_dir / "parcellation.bin"))
        with open(mesh_dir / "roi_names.json", "w") as f:
            json.dump(roi_names, f)
    except Exception as e:
        print(f"  Parcellation skipped: {e}")

print(f"  Mesh: {vertices.shape[0]} vertices, {faces_arr.shape[0]} faces")

In [ ]:
with timed_phase(bench, "9b. Export predictions",
                 notes="Binary predictions + metadata JSON"):
    pred_dir = OUTPUT_DIR / "predictions"
    pred_dir.mkdir(parents=True, exist_ok=True)
    preds_norm.tofile(str(pred_dir / "predictions.bin"))

    n_timesteps = preds_norm.shape[0]
    metadata = {
        "nTimesteps": int(n_timesteps),
        "nVertices": int(preds_norm.shape[1]),
        "trSeconds": 1.0,
        "vmin": 0.5,
        "alphaScale": 0.2,
    }
    with open(pred_dir / "metadata.json", "w") as f:
        json.dump(metadata, f, indent=2)

print(f"  Predictions: {preds_norm.shape}")

In [ ]:
with timed_phase(bench, "9c. Export stimulus metadata",
                 notes="segments.json + video copy + thumbnails"):
    stim_dir = OUTPUT_DIR / "stimulus"
    stim_dir.mkdir(parents=True, exist_ok=True)

    # Segments JSON
    segments_data = []
    for seg in all_segments:
        entry = {"time": float(seg.start), "hasEvents": len(seg.ns_events) > 0, "words": []}
        for ev in seg.ns_events:
            if ev.__class__.__name__ == "Word":
                entry["words"].append({
                    "text": str(ev.text),
                    "start": float(ev.start),
                    "end": float(ev.start + ev.duration),
                })
        segments_data.append(entry)
    with open(stim_dir / "segments.json", "w") as f:
        json.dump(segments_data, f, indent=2)

    # Copy video
    shutil.copy2(str(video_path), str(stim_dir / "media.mp4"))

    # Thumbnails
    thumb_dir = stim_dir / "thumbnails"
    thumb_dir.mkdir(exist_ok=True)
    try:
        from moviepy import VideoFileClip
        from PIL import Image
        clip = VideoFileClip(str(video_path))
        for i, seg in enumerate(all_segments):
            t = seg.start
            if 0 <= t < clip.duration:
                frame = clip.get_frame(t)
                Image.fromarray(frame).save(str(thumb_dir / f"frame_{i:05d}.jpg"), "JPEG", quality=80)
        clip.close()
    except Exception as e:
        print(f"  Thumbnails skipped: {e}")

print(f"  Segments: {len(segments_data)}, video copied, thumbnails extracted")

## 12. Zip + download

In [ ]:
with timed_phase(bench, "10. Zip output"):
    zip_path = shutil.make_archive("viewer_data", "zip", ".", "viewer_data")

print(f"Created {zip_path} ({Path(zip_path).stat().st_size / 1e6:.1f} MB)")

## 13. Benchmark results

In [ ]:
# Print the full summary table
bench.summary()

# Save to JSON
results_path = Path("benchmark_results.json")
with open(results_path, "w") as f:
    json.dump(bench.to_dict(), f, indent=2)

print(f"\nResults saved to {results_path}")
print("\nFull results:")
print(json.dumps(bench.to_dict(), indent=2))

## 14. Download results

Downloads both the benchmark JSON and the viewer data zip.

In [ ]:
from google.colab import files

# Download benchmark results
files.download("benchmark_results.json")

# Download viewer data (same as export notebook)
files.download("viewer_data.zip")